# Word2vec from Scratch: A Small Corpus, Weak Analogies

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/nlp/word2vec_from_scratch.ipynb)

Train one NumPy epoch, compare three gensim seeds per arm, and inspect analogy coverage.

Companion tutorial: [sesen.ai](https://sesen.ai/blog/word2vec-from-scratch).

CPU only. Run all cells in a fresh kernel. All examples use measured algorithms; saved blog results may differ across library versions or hardware.

In [ ]:
import importlib.util, importlib.metadata, subprocess, sys, os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ.setdefault("MPLCONFIGDIR", "/tmp/batch11_matplotlib")
from pathlib import Path
WORK = Path.cwd() / "word2vec_work"
WORK.mkdir(exist_ok=True)
requirements = {'numpy': 'numpy', 'scipy': 'scipy', 'matplotlib': 'matplotlib', 'sklearn': 'scikit-learn', 'gensim': 'gensim==4.4.0'}
install = []
for module, requirement in requirements.items():
    distribution, _, pinned = requirement.partition("==")
    if importlib.util.find_spec(module) is None:
        install.append(requirement)
    elif pinned and importlib.metadata.version(distribution) != pinned:
        install.append(requirement)
if install:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *install])
print({requirement.split("==")[0]: importlib.metadata.version(requirement.split("==")[0])
       for requirement in requirements.values()})


In [ ]:
import hashlib, urllib.request
from pathlib import Path
DATA = Path.cwd() / "batch11_data"
DATA.mkdir(exist_ok=True)
def download(url, filename, expected=None):
    path = DATA / filename
    if not path.exists():
        path.write_bytes(urllib.request.urlopen(url, timeout=90).read())
    if expected is not None:
        assert hashlib.sha256(path.read_bytes()).hexdigest() == expected
    return path
download("https://mattmahoney.net/dc/text8.zip", "text8.zip",
         "a6640522afe85d1963ad56c05b0ede0a0c000dddc9671758a6cc09b7a38e5232")
download("https://raw.githubusercontent.com/tmikolov/word2vec/master/questions-words.txt",
         "questions-words.txt", "8c29b3332afc46f3fb8be04cb5297bf96f39aa7131272dff57869b4485b22a36")


## Algorithm implementation

The following definitions are complete and do not import project-local modules.

In [ ]:
"""NumPy skip-gram with negative sampling, including repeated-index updates."""

from collections import Counter

import numpy as np
from scipy.special import expit


def vocabulary(words, min_count=5):
    counts = Counter(words)
    vocab = sorted((w for w in counts if counts[w] >= min_count),
                   key=lambda w: (-counts[w], w))
    lookup = {w: i for i, w in enumerate(vocab)}
    ids = np.array([lookup[w] for w in words if w in lookup], dtype=np.int32)
    frequencies = np.array([counts[w] for w in vocab], dtype=np.float64)
    return vocab, lookup, ids, frequencies


def subsample(ids, counts, rng, threshold=1e-3):
    if threshold == 0:
        return ids.copy()
    frequency = counts / counts.sum()
    keep = np.minimum(1, (np.sqrt(frequency / threshold) + 1) * threshold / frequency)
    return ids[rng.random(len(ids)) < keep[ids]]


def pairs(ids, rng, window=5):
    """Each centre gets a uniform random radius, as in dynamic-window SGNS."""
    radius = rng.integers(1, window + 1, len(ids))
    centres, contexts = [], []
    for offset in range(1, window + 1):
        left = np.arange(len(ids) - offset)
        right = left + offset
        use = radius[left] >= offset
        centres.append(ids[left[use]])
        contexts.append(ids[right[use]])
        use = radius[right] >= offset
        centres.append(ids[right[use]])
        contexts.append(ids[left[use]])
    return np.concatenate(centres), np.concatenate(contexts)


def sgns_loss_gradient(v, u, labels):
    scores = u @ v
    loss = np.logaddexp(0, scores) - labels * scores
    error = expit(scores) - labels
    return float(loss.sum()), error @ u, error[:, None] * v


def gradient_check():
    rng = np.random.default_rng(9)
    v = rng.normal(size=4)
    u = rng.normal(size=(6, 4))
    labels = np.array([1, 0, 0, 0, 0, 0])
    _, gv, gu = sgns_loss_gradient(v, u, labels)
    errors = []
    for array, analytic in [(v, gv), (u, gu)]:
        for index in np.ndindex(array.shape):
            original = array[index]
            array[index] = original + 1e-5
            plus = sgns_loss_gradient(v, u, labels)[0]
            array[index] = original - 1e-5
            minus = sgns_loss_gradient(v, u, labels)[0]
            array[index] = original
            errors.append(abs((plus - minus) / 2e-5 - analytic[index]))
    return float(max(errors))


def objective(V, U, centres, targets):
    scores = np.einsum('bd,bkd->bk', V[centres], U[targets])
    return float((np.logaddexp(0, -scores[:, 0]) +
                  np.logaddexp(0, scores[:, 1:]).sum(axis=1)).mean())


def train_sgns(ids, counts, dimension=100, window=5, negatives=5,
               sample=1e-3, exponent=0.75, seed=0, batch_size=256,
               checkpoints=10, progress=None):
    rng = np.random.default_rng(seed)
    retained = subsample(ids, counts, rng, sample)
    centre, context = pairs(retained, rng, window)
    order = rng.permutation(len(centre))
    probability = counts ** exponent
    probability /= probability.sum()
    V = rng.uniform(-0.5 / dimension, 0.5 / dimension,
                    (len(counts), dimension)).astype(np.float32)
    U = np.zeros_like(V)
    probe = rng.choice(len(centre), size=min(10000, len(centre)), replace=False)
    pc = centre[probe]
    pt = np.column_stack([context[probe], rng.choice(len(counts),
                           size=(len(probe), negatives), p=probability)])
    history = [(0.0, objective(V, U, pc, pt))]
    next_checkpoint = 1
    for start in range(0, len(order), batch_size):
        ix = order[start:start + batch_size]
        c = centre[ix]
        targets = np.column_stack([context[ix], rng.choice(len(counts),
                                  size=(len(ix), negatives), p=probability)])
        # Copies freeze both matrices while forming this batch's gradients.
        v, u = V[c].copy(), U[targets].copy()
        scores = np.einsum('bd,bkd->bk', v, u)
        error = expit(scores)
        error[:, 0] -= 1
        learning_rate = 0.025 * (1 - start / len(order)) + 0.0001 * start / len(order)
        np.add.at(V, c, -learning_rate * np.einsum('bk,bkd->bd', error, u))
        np.add.at(U, targets.ravel(),
                  (-learning_rate * error[:, :, None] * v[:, None, :]).reshape(-1, dimension))
        fraction = min(1.0, (start + batch_size) / len(order))
        if fraction >= next_checkpoint / checkpoints:
            history.append((fraction, objective(V, U, pc, pt)))
            if progress:
                progress(fraction, history[-1][1])
            next_checkpoint += 1
    return V, U, history, len(centre)


def normalise(vectors):
    return vectors / np.maximum(np.linalg.norm(vectors, axis=1, keepdims=True), 1e-12)


def neighbours(word, vectors, lookup, vocab, k=5):
    scores = normalise(vectors) @ normalise(vectors)[lookup[word]]
    scores[lookup[word]] = -np.inf
    top = np.argsort(scores)[-k:][::-1]
    return [(vocab[i], float(scores[i])) for i in top]


def analogy_score(vectors, lookup, questions, exclude=True):
    unit = normalise(vectors)
    correct = 0
    for start in range(0, len(questions), 128):
        q = questions[start:start + 128]
        a, b, c, d = q.T
        query = normalise(unit[b] - unit[a] + unit[c])
        scores = query @ unit.T
        if exclude:
            scores[np.arange(len(q))[:, None], q[:, :3]] = -np.inf
        correct += int((scores.argmax(axis=1) == d).sum())
    return correct / len(questions) if len(questions) else float('nan')


## Reproducible measurements

These cells run the experiments reported in the post. The synthetic datasets have fixed seeds and known targets. The result files preserve individual replications, settings and timings.

In [ ]:
"""Post 108: one NumPy run, nine reference runs, and a sparse PPMI check."""
import hashlib
import json
import os
from pathlib import Path
import time
import zipfile

import numpy as np
from gensim.models import Word2Vec
from scipy import sparse
from sklearn.decomposition import TruncatedSVD


ROOT = WORK
OUT = ROOT / 'scripts/w2v_results'
OUT.mkdir(parents=True, exist_ok=True)

t0 = time.perf_counter()
archive = DATA / 'text8.zip'
assert hashlib.sha256(archive.read_bytes()).hexdigest() == 'a6640522afe85d1963ad56c05b0ede0a0c000dddc9671758a6cc09b7a38e5232'
with zipfile.ZipFile(archive) as z:
    words = z.read('text8').decode().split()[:2000000]
vocab, lookup, ids, counts = vocabulary(words)
questions = []
categories = []
category = ''
for line in (DATA / 'questions-words.txt').read_text().splitlines():
    if line.startswith(':'):
        category = line[2:]
    else:
        row = line.lower().split()
        if len(row) == 4 and all(w in lookup for w in row):
            questions.append([lookup[w] for w in row])
            categories.append(category)
questions = np.array(questions)
np.savez_compressed(OUT / 'corpus.npz', ids=ids, counts=counts, vocab=np.array(vocab), questions=questions)
results = {'tokens':len(words), 'vocabulary':len(vocab), 'questions':len(questions),
           'gradient_max_error': gradient_check(), 'archive_sha256':hashlib.sha256(archive.read_bytes()).hexdigest()}
print('data',results,flush=True)
start = time.perf_counter()
V, U, history, n_pairs = train_sgns(ids, counts, progress=lambda p,l:print('NumPy',round(p,2),round(l,4),round(time.perf_counter()-start,1),flush=True))
results['numpy_seconds'] = time.perf_counter()-start
results['pairs'] = n_pairs
results['history'] = history
results['numpy_accuracy'] = analogy_score(V,lookup,questions)
results['numpy_accuracy_unrestricted'] = analogy_score(V,lookup,questions,exclude=False)
np.savez_compressed(OUT / 'vectors.npz', vectors=V, output_vectors=U, vocab=np.array(vocab), counts=counts)
print('NumPy accuracy',results['numpy_accuracy'],flush=True)
sentences = [words[i:i+1000] for i in range(0,len(words),1000)]
arms = {}
reference = None
for arm,sample,exponent in [('baseline',1e-3,0.75),('no_subsampling',0,0.75),('exponent_1',1e-3,1.0)]:
    runs=[]
    for seed in range(3):
        start=time.perf_counter()
        m=Word2Vec(sentences,vector_size=100,window=5,min_count=5,sg=1,hs=0,negative=5,
                   sample=sample,ns_exponent=exponent,seed=seed,workers=1,epochs=1,
                   alpha=0.025,min_alpha=0.0001,shrink_windows=True,sorted_vocab=1)
        aligned=np.array([m.wv[w] for w in vocab])
        runs.append({'seed':seed,'accuracy':analogy_score(aligned,lookup,questions),
                     'seconds':time.perf_counter()-start})
        print(arm,runs[-1],flush=True)
        if arm=='baseline' and seed==0:
            reference=aligned
            np.savez_compressed(OUT / 'gensim_vectors.npz',vectors=aligned,
                               output_vectors=np.array([m.syn1neg[m.wv.key_to_index[w]] for w in vocab]))
    arms[arm]=runs
results['arms']=arms
probes=['king','queen','france','paris','computer','music']
results['neighbours']={w:{'numpy':neighbours(w,V,lookup,vocab),
                          'gensim':neighbours(w,reference,lookup,vocab)} for w in probes}
# Sparse cross-check restricted to frequent words; no dense V x V allocation.
n_ppmi=min(5000,len(vocab))
rng=np.random.default_rng(22)
c,o=pairs(ids[:250000],rng,window=2)
mask=(c<n_ppmi)&(o<n_ppmi)
co=sparse.coo_matrix((np.ones(mask.sum()),(c[mask],o[mask])),shape=(n_ppmi,n_ppmi)).tocsr()
co.sum_duplicates()
coo=co.tocoo()
row=np.asarray(co.sum(axis=1)).ravel()
col=np.asarray(co.sum(axis=0)).ravel()
values=np.maximum(0,np.log(coo.data*co.sum()/(row[coo.row]*col[coo.col])))
ppmi=sparse.coo_matrix((values,(coo.row,coo.col)),shape=co.shape).tocsr()
svd=TruncatedSVD(n_components=100,random_state=0).fit_transform(ppmi)
results['ppmi_neighbours']={w:neighbours(w,svd,lookup,vocab[:n_ppmi]) for w in probes if lookup[w]<n_ppmi}
results['ppmi_tokens']=250000
results['total_seconds']=time.perf_counter()-t0
(OUT/'results.json').write_text(json.dumps(results,indent=2))
print('DONE',results['total_seconds'],flush=True)

## Figures

Every experimental chart below is generated from the preceding run. The GIFs show actual operations or recorded gate activations.

In [ ]:
"""Figures derived from the saved Batch 11 measurements."""
import json
from pathlib import Path
import sys

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
import numpy as np
from sklearn.decomposition import PCA

ROOT = WORK
IMAGES = WORK / 'figures'
IMAGES.mkdir(exist_ok=True)
plt.rcParams.update({'font.size':11,'axes.spines.top':False,'axes.spines.right':False,
                     'figure.facecolor':'white','axes.grid':True,'grid.alpha':.15})
TEAL='#168f91';AMBER='#d99120';BLUE='#426887'

def save(fig,slug,name):
    fig.tight_layout()
    fig.savefig(IMAGES/f'{slug}-{name}.webp',dpi=150,bbox_inches='tight',facecolor='white')
    plt.close(fig)

def word2vec():
    slug='word2vec-from-scratch'
    r=json.loads((ROOT/'scripts/w2v_results/results.json').read_text())
    data=np.load(ROOT/'scripts/w2v_results/vectors.npz');ref=np.load(ROOT/'scripts/w2v_results/gensim_vectors.npz')
    corpus=np.load(ROOT/'scripts/w2v_results/corpus.npz')
    rng=np.random.default_rng(44)
    c,o=pairs(corpus['ids'][:20000],rng)
    ix=rng.choice(len(c),5000,replace=False)
    q=corpus['counts']**.75;q/=q.sum()
    targets=np.column_stack([o[ix],rng.choice(len(q),size=(len(ix),5),p=q)])
    losses=[objective(data['vectors'],data['output_vectors'],c[ix],targets),
            objective(ref['vectors'],ref['output_vectors'],c[ix],targets)]
    r['common_probe_loss']={'numpy':losses[0],'gensim':losses[1],
                            'description':'5000 fixed pairs from first 20000 unfiltered tokens; five shared negatives'}
    (ROOT/'scripts/w2v_results/results.json').write_text(json.dumps(r,indent=2))
    fig,axes=plt.subplots(1,2,figsize=(10,4))
    h=np.array(r['history']);axes[0].plot(h[:,0],h[:,1],color=TEAL,marker='o')
    axes[0].set(xlabel='Fraction of one epoch',ylabel='SGNS loss per positive pair',title='NumPy fixed training probe')
    axes[1].bar(['NumPy','gensim'],losses,color=[TEAL,BLUE]);axes[1].set(ylabel='SGNS loss per positive pair',title='Shared post-training probe')
    save(fig,slug,'loss')
    fig,ax=plt.subplots(figsize=(8,4))
    arms=['baseline','no_subsampling','exponent_1']
    vals=[[100*a['accuracy'] for a in r['arms'][arm]] for arm in arms]
    ax.bar(['Baseline','Subsampling off','Noise exponent 1.0'],[np.mean(v) for v in vals],
           yerr=[np.std(v,ddof=1) for v in vals],capsize=6,color=[TEAL,AMBER,BLUE])
    ax.axhline(100*r['numpy_accuracy'],color=BLUE,linestyle='--',label='NumPy, one seed')
    ax.set(ylabel='Analogy accuracy (%)',title=f'{r["questions"]:,} in-vocabulary questions; gensim, 3 seeds');ax.legend()
    save(fig,slug,'ablation')
    vocab=data['vocab'].tolist();lookup={w:i for i,w in enumerate(vocab)}
    groups=[['france','paris','germany','berlin','england','london'],['king','queen','man','woman'],['computer','software','music','song']]
    words=sum(groups,[])
    pca=PCA(2).fit(data['vectors'])
    coordinates=pca.transform(data['vectors'][[lookup[w] for w in words]])
    fig,ax=plt.subplots(figsize=(9,5))
    for group,colour in zip(groups,[TEAL,AMBER,BLUE]):
        for word in group:
            i=words.index(word);xy=coordinates[i];ax.scatter(*xy,color=colour)
            offsets={'paris':(-30,12),'queen':(5,-15),'king':(-5,12),
                     'london':(5,-14),'germany':(-45,12),'england':(-15,-16)}
            ax.annotate(word,xy,xytext=offsets.get(word,(4,4)),textcoords='offset points',fontsize=10)
    king=coordinates[words.index('king')];man=coordinates[words.index('man')];woman=coordinates[words.index('woman')]
    target=king-man+woman
    ax.annotate('',target,king,arrowprops={'arrowstyle':'->','color':AMBER})
    ax.scatter(*target,marker='x',color=AMBER,s=80)
    ax.annotate('king - man + woman',target,xytext=(-145,10),textcoords='offset points',fontsize=9)
    ax.set(xlabel='Principal component 1',ylabel='Principal component 2',title='Two-dimensional projection; analogy endpoint is a computed vector')
    save(fig,slug,'pca')
    fig,ax=plt.subplots(figsize=(10,4));ax.axis('off')
    rows=[]
    for word,methods in r['neighbours'].items():
        rows.append([word,', '.join(w for w,_ in methods['numpy'][:3]),', '.join(w for w,_ in methods['gensim'][:3])])
    table=ax.table(cellText=rows,colLabels=['Probe','NumPy: nearest 3','gensim: nearest 3'],cellLoc='left',loc='center',colWidths=[.12,.44,.44])
    table.auto_set_font_size(False);table.set_fontsize(10);table.scale(1,2)
    ax.set_title('Input embeddings, cosine similarity, self excluded')
    save(fig,slug,'neighbours')
    sentence='the king and the queen rule the country together'.split()
    fig,ax=plt.subplots(figsize=(10,2.5))
    def update(frame):
        ax.clear();ax.axis('off');centre=frame%len(sentence)
        for i,word in enumerate(sentence):
            colour=AMBER if i==centre else TEAL if abs(i-centre)<=2 else '#d8e0e5'
            ax.text(i,.5,word,ha='center',bbox={'boxstyle':'round,pad=.4','fc':colour,'alpha':.7})
            if i!=centre and abs(i-centre)<=2:
                ax.annotate('',(i,.78),(centre,.78),arrowprops={'arrowstyle':'->','color':TEAL,'connectionstyle':'arc3,rad=-.25'})
        ax.set(xlim=(-.6,len(sentence)-.4),ylim=(0,1.2),title='Centre word (amber) creates observed context pairs (teal), radius 2')
    FuncAnimation(fig,update,frames=12).save(IMAGES/f'{slug}-window.gif',writer=PillowWriter(fps=3),dpi=100)
    plt.close(fig)

def kelly():
    slug='kelly-criterion-python';r=json.loads((ROOT/'scripts/kelly_results/results.json').read_text())
    f=np.linspace(0,.45,301);p=r['p'];b=r['b']
    fig,axes=plt.subplots(1,2,figsize=(10,4))
    axes[0].plot(f,p*np.log1p(b*f)+(1-p)*np.log1p(-f),color=TEAL)
    axes[0].axvline(.1,color=AMBER,label='Kelly');axes[0].axvline(.2,color=BLUE,linestyle='--',label='2 x Kelly')
    axes[0].axhline(0,color='gray',linewidth=.8);axes[0].set(xlabel='Bet fraction',ylabel='Expected log growth per bet',title='Binary bet: p = 0.55, b = 1');axes[0].legend()
    f=np.linspace(0,6,301);axes[1].plot(f,f*.1-.5*f*f*.2**2,color=TEAL)
    axes[1].axvline(2.5,color=AMBER);axes[1].axvline(5,color=BLUE,linestyle='--');axes[1].axhline(0,color='gray',linewidth=.8)
    axes[1].set(xlabel='Asset exposure / wealth',ylabel='Annual excess log growth',title='Continuous model: mean 10%, volatility 20%')
    save(fig,slug,'growth')
    fig,axes=plt.subplots(2,2,figsize=(10,7))
    for ax,(scale,fan) in zip(axes.ravel(),r['fans'].items()):
        q=np.array(fan);t=np.arange(q.shape[1])/10
        ax.fill_between(t,q[0],q[2],color=TEAL,alpha=.2);ax.plot(t,q[1],color=TEAL)
        ax.set(yscale='log',xlabel='Years',ylabel='Wealth / initial wealth',title=f'{scale} x true Kelly, 10-90% band')
    save(fig,slug,'wealth')
    fig,ax=plt.subplots(figsize=(8,4))
    ax.boxplot([r['drawdowns'][s] for s in ['0.25','0.5','1','2']],tick_labels=['Quarter','Half','Full','Double'],showfliers=False)
    ax.set(ylabel='Maximum drawdown',title='10,000 paths; 2,500 daily observations, known inputs')
    save(fig,slug,'drawdown')
    fig,axes=plt.subplots(1,2,figsize=(10,4))
    axes[0].hist(r['estimated_kelly'],bins=40,color=TEAL);axes[0].axvline(r['true_kelly'],color=AMBER,label='True Kelly')
    axes[0].set(xlabel='Estimated Kelly (clipped to 0-10)',ylabel='Trials',title='Mean estimated from five years');axes[0].legend()
    scales=['0.25','0.5','1','2'];axes[1].bar(scales,[r['estimation'][s]['median_terminal'] for s in scales],color=[BLUE,TEAL,AMBER,BLUE])
    axes[1].set(xlabel='Fraction of estimated Kelly',ylabel='Median terminal wealth',title='Independent ten-year future')
    save(fig,slug,'estimation')
    fig,axes=plt.subplots(2,1,figsize=(10,7),sharex=True)
    dates=np.array(r['dates'],dtype='datetime64[D]')
    for name,colour in [('constant',BLUE),('garch',TEAL)]:
        axes[0].plot(dates,r['backtest'][name]['wealth'],color=colour,label=name)
        axes[1].plot(dates,r['backtest'][name]['exposure'],color=colour,label=name,linewidth=.8)
    axes[0].set(yscale='log',ylabel='Wealth / initial wealth',title='S&P 500 price returns, assumed 5% premium, costs and leverage financing')
    axes[1].set(ylabel='Exposure',xlabel='Forecast origin');axes[0].legend()
    save(fig,slug,'backtest')

def hnsw():
    slug='hnsw-python';r=json.loads((ROOT/'scripts/hnsw_results/results.json').read_text())
    fig,ax=plt.subplots(figsize=(8,4))
    ax.plot([v['dimension'] for v in r['kd']],[100*v['fraction'] for v in r['kd']],color=TEAL,marker='o')
    ax.axhline(100,color=AMBER,linestyle='--',label='Brute-force distance count')
    ax.set(xlabel='Dimensions',ylabel='Points evaluated (%)',title='Exact KD-tree: 4,000 Gaussian points, 100 independent queries');ax.legend()
    save(fig,slug,'kdtree')
    fig,axes=plt.subplots(1,2,figsize=(10,4));s=r['sweep']
    axes[0].plot([100*v['distance_fraction'] for v in s],[v['recall'] for v in s],color=TEAL,marker='o')
    for v in s:axes[0].annotate(str(v['ef']),(100*v['distance_fraction'],v['recall']),xytext=(3,-12),textcoords='offset points',fontsize=9)
    axes[0].set(xlabel='Exact distance budget (%)',ylabel='Recall@10',ylim=(.8,1.015),title='NumPy index; labels are ef_search')
    for name,key,colour in [('Python','recall',TEAL),('hnswlib','hnswlib_recall',BLUE)]:
        axes[1].plot([v['ef'] for v in s],[v[key] for v in s],marker='o',label=name,color=colour)
    axes[1].set(xlabel='ef_search',ylabel='Recall@10',xscale='log',ylim=(.8,1.015),title='Matched M = 12, ef_construction = 80');axes[1].legend()
    save(fig,slug,'recall')
    fig,ax=plt.subplots(figsize=(8,4))
    ax.plot([v['ef'] for v in s],[v['qps'] for v in s],marker='o',label='Python',color=TEAL)
    ax.plot([v['ef'] for v in s],[v['hnswlib_qps'] for v in s],marker='o',label='hnswlib',color=BLUE)
    ax.axhline(r['exact_qps'],color=AMBER,linestyle='--',label='Vectorised exact (batched)')
    ax.set(xscale='log',yscale='log',xlabel='ef_search',ylabel='Queries per second',title='Timing includes distance calculations; 100 held-out queries');ax.legend()
    save(fig,slug,'speed')
    fig,ax=plt.subplots(figsize=(11,4));ax.axis('off');rows=[]
    for word,d in r['neighbours'].items():
        rows.append([word,', '.join(d['exact'][:4]),', '.join(d['hnsw'][:4])])
    table=ax.table(cellText=rows,colLabels=['Probe','Exact: nearest 4','HNSW: nearest 4'],loc='center',cellLoc='left',colWidths=[.12,.44,.44]);table.auto_set_font_size(False);table.set_fontsize(10);table.scale(1,2)
    ax.set_title('Probe queries shown separately; ef_search = 80, self excluded')
    save(fig,slug,'neighbours')
    rng=np.random.default_rng(3);x=rng.normal(size=(250,2)).astype(np.float32)
    index=HNSW(x,M=5,ef_construction=30,seed=2).build();q=np.array([1.1,.6],dtype=np.float32)
    found,_=index.query(q,k=5,ef=15);evaluated=list(index.cache)
    fig,axes=plt.subplots(1,3,figsize=(11,4))
    def update(frame):
        seen=set(evaluated[:max(1,int(len(evaluated)*(frame+1)/12))])
        for layer,ax in zip([2,1,0],axes):
            ax.clear();nodes=index.layers[layer]
            for a,links in nodes.items():
                for b in links:ax.plot(*x[[a,b]].T,color='#c5d5dc',linewidth=.5,zorder=0)
            ids=list(nodes);ax.scatter(*x[ids].T,c=[AMBER if i in seen else TEAL for i in ids],s=25)
            ax.scatter(*q,marker='*',color=BLUE,s=120)
            if frame==11:
                present=[i for i in found if i in nodes]
                if present:ax.scatter(*x[present].T,facecolors='none',edgecolors=BLUE,s=80)
            ax.set(xlim=(-3,3),ylim=(-3,3),title=f'Layer {layer}');ax.set_aspect('equal')
        fig.suptitle('Actual HNSW evaluations: amber = visited, star = query, rings = results')
    FuncAnimation(fig,update,frames=12).save(IMAGES/f'{slug}-search.gif',writer=PillowWriter(fps=3),dpi=100);plt.close(fig)

def dml():
    slug='double-machine-learning-python';r=json.loads((ROOT/'scripts/dml_results/results.json').read_text())
    methods=['naive','ols','plugin','no_crossfit','dml'];labels=['Unadjusted OLS','Linear adjustment','Plug-in ML','DML, in sample','Cross-fitted DML']
    fig,axes=plt.subplots(1,2,figsize=(11,4))
    axes[0].boxplot([[v[m] for v in r['runs']] for m in methods],tick_labels=labels,vert=False,showfliers=True)
    axes[0].axvline(.5,color=TEAL,linestyle='--',label='True effect');axes[0].set(xlabel='Effect estimate',title='50 independent synthetic datasets');axes[0].legend()
    names=['no_crossfit','dml'];axes[1].bar(['In-sample DML','Cross-fitted DML'],[100*r['summary'][m]['coverage'] for m in names],color=[AMBER,TEAL]);axes[1].axhline(95,color=BLUE,linestyle='--');axes[1].set(ylim=(0,100),ylabel='95% interval coverage (%)',title='Monte Carlo uncertainty: 50 replications')
    save(fig,slug,'estimates')
    residual=np.load(ROOT/'scripts/dml_results/residuals.npz');d=residual['d'];y=residual['y']
    theta=float(d@y/(d@d));fig,ax=plt.subplots(figsize=(8,4));ax.scatter(d,y,s=8,alpha=.15,color=TEAL)
    grid=np.linspace(d.min(),d.max(),100);ax.plot(grid,theta*grid,color=AMBER,label=f'Slope = {theta:.3f}');ax.set(xlabel='Treatment minus predicted treatment',ylabel='Outcome minus predicted outcome',title='Out-of-fold residuals, final replication');ax.legend();save(fig,slug,'residuals')
    real=r['real'];fig,ax=plt.subplots(figsize=(8,3));ax.errorbar([real['theta'],real['doubleml_theta']],[1,0],xerr=1.96*np.array([real['se'],real['doubleml_se']]),fmt='o',color=TEAL,capsize=5)
    ax.set(yticks=[0,1],yticklabels=['DoubleML','From scratch'],xlabel='401(k) eligibility effect on net financial assets ($)',title='Same five folds and nuisance learners; 95% intervals');save(fig,slug,'401k')

def lstm():
    slug='lstm-from-scratch';r=json.loads((ROOT/'scripts/lstm_results/results.json').read_text())
    fig,ax=plt.subplots(figsize=(8,4))
    for architecture,bias,colour in [('rnn',None,BLUE),('lstm',1,AMBER),('lstm',3,TEAL)]:
        lengths=[10,25,50,100] if bias!=3 else [50,100]
        runs=r['bias3'] if bias==3 else r['runs']
        values=[np.array([v['accuracy'] for v in runs if v['architecture']==architecture and v['length']==t]) for t in lengths]
        means=np.array([v.mean() for v in values]);sd=np.array([v.std(ddof=1) for v in values])
        label='RNN' if architecture=='rnn' else f'LSTM, forget bias {bias}'
        ax.plot(lengths,means,color=colour,marker='o',label=label);ax.fill_between(lengths,np.maximum(.0,means-sd),np.minimum(1,means+sd),color=colour,alpha=.15)
    ax.axhline(.5,color='gray',linestyle='--');ax.set(xlabel='Sequence length',ylabel='Test accuracy',ylim=(.35,1.03),title='Recall the first symbol: 3 seeds, bands = one seed SD');ax.legend();save(fig,slug,'accuracy')
    fig,ax=plt.subplots(figsize=(8,4))
    for architecture,bias,colour in [('rnn',None,BLUE),('lstm',1,AMBER),('lstm',3,TEAL)]:
        runs=r['bias3'] if bias==3 else r['runs'];v=next(v for v in runs if v['architecture']==architecture and v['length']==100 and v['seed']==0)
        gradient=np.array(v['gradient']);ax.semilogy(np.arange(99,-1,-1),np.maximum(gradient,1e-30),color=colour,label='RNN' if bias is None else f'LSTM, forget bias {bias}')
    ax.set(xlabel='Lag from final output',ylabel='Input gradient norm',title='Trained models, one held-out sequence, seed 0');ax.legend();save(fig,slug,'gradient')
    fig,ax=plt.subplots(figsize=(8,4))
    for bias,colour in [(0,BLUE),(1,AMBER),(3,TEAL)]:
        runs=r['ablation'] if bias==0 else r['bias3'] if bias==3 else r['runs']
        records=[v for v in runs if v['architecture']=='lstm' and v['length']==100]
        h=np.array([v['history'] for v in records]);ax.plot(h[0,:,0],h[:,:,2].mean(axis=0),color=colour,label=f'Forget bias {bias}')
    ax.set(xlabel='Training steps',ylabel='Test accuracy',xlim=(0,300),title='T = 100, first 300 of 1,000 steps, three seeds per bias');ax.legend();save(fig,slug,'bias')
    trace=r['gate_trace'];gates=np.array(trace['gates']);states=np.array(trace['cell'])
    fig,axes=plt.subplots(2,1,figsize=(9,5),sharex=True)
    def update(frame):
        stop=min(len(gates),int((frame+1)*len(gates)/12))
        for ax in axes:ax.clear()
        for j,(name,colour) in enumerate(zip(['Input','Forget','Candidate','Output'],[TEAL,AMBER,BLUE,'#8a5693'])):
            axes[0].plot(np.arange(stop),gates[:stop,j],label=name,color=colour)
        axes[0].legend(ncol=4,loc='lower left');axes[0].set(ylim=(-1,1.05),ylabel='Mean gate activation')
        padding=max(.05,.05*np.ptp(states))
        axes[1].plot(np.arange(stop),states[:stop],color=TEAL);axes[1].set(xlim=(0,100),ylim=(states.min()-padding,states.max()+padding),xlabel='Sequence step',ylabel='Mean cell state')
        fig.suptitle('Trained LSTM, forget bias 3: one actual held-out sequence')
    FuncAnimation(fig,update,frames=12).save(IMAGES/f'{slug}-gates.gif',writer=PillowWriter(fps=3),dpi=100);plt.close(fig)


word2vec()
from IPython.display import display, Image
for path in sorted(IMAGES.glob("word2vec-from-scratch-*")):
    if path.suffix in {".webp", ".gif"}:
        display(Image(filename=str(path)))

In [ ]:
saved = np.load(WORK / "scripts/w2v_results/vectors.npz")
np.savez_compressed("word2vec_vectors.npz", vectors=saved["vectors"],
                    vocab=saved["vocab"], counts=saved["counts"])
print("Saved word2vec_vectors.npz for the HNSW notebook.")

## Exercises

1. Compare restricted and unrestricted analogy scores.
2. Train with a smaller window and inspect the change in neighbours.
3. Compare V, U and their mean as downstream embeddings; keep the evaluation questions fixed.